In [ ]:
"""
Adds structural features to the existing engineered features, for the
training set and the temporal test set.

These features measure the SHAPE of the text (length, sentence structure,
compulsion-word density, list-like structure) rather than specific wording,
so they should generalize better to new jailbreak phrasing than the
lexicon-based (lex_*) features do.

Outputs:
    Data/struct/features_main_struct.csv
    Data/struct/features_test_temporal_struct.csv
    Data/struct/features_test_obfuscated_struct.csv
    Data/struct/features_test_notinject_struct.csv

Each output file = original engineered features + new struct_* columns.
"""

import os
import re
import pandas as pd

RE_SENT = re.compile(r"(?<=[.!?\n])\s+")
RE_WORD = re.compile(r"[a-z']+")

# Words that show up heavily in "compulsion" / rule-forcing language,
# regardless of the specific jailbreak persona/style used
COMPULSION_WORDS = {
    "must", "always", "never", "will", "shall", "cannot", "can't",
    "should", "required", "mandatory", "forbidden", "not allowed",
    "have to", "has to", "need to", "no matter", "at all costs",
    "under no circumstances", "without exception"
}

# Patterns that indicate a numbered/bulleted rule-list structure
RE_NUMBERED_LIST = re.compile(r"(?:^|\n)\s*(?:\d+[\.\)]|[-*•])\s+", re.MULTILINE)


def _structural_row(raw):
    text = raw if isinstance(raw, str) else ""
    n_chars = max(len(text), 1)

    sentences = [s for s in RE_SENT.split(text) if s.strip()] or [""]
    words = RE_WORD.findall(text.lower()) or [""]

    n_sentences = len(sentences)
    avg_sentence_len = sum(len(RE_WORD.findall(s.lower())) for s in sentences) / n_sentences

    # Count compulsion words/phrases (some are multi-word, so check substrings too)
    low = text.lower()
    compulsion_hits = sum(low.count(w) for w in COMPULSION_WORDS)
    compulsion_density = compulsion_hits / n_sentences

    # Numbered/bulleted list density
    list_items = len(RE_NUMBERED_LIST.findall(text))

    # ALL-CAPS word ratio (shouted emphasis, common in DAN-style prompts)
    caps_words = sum(1 for w in re.findall(r"[A-Za-z']+", text) if w.isupper() and len(w) > 2)
    caps_word_ratio = caps_words / max(len(re.findall(r"[A-Za-z']+", text)), 1)

    return {
        "struct_n_sentences": n_sentences,
        "struct_avg_sentence_len": avg_sentence_len,
        "struct_compulsion_density": compulsion_density,
        "struct_list_items": list_items,
        "struct_list_density": list_items / n_sentences,
        "struct_caps_word_ratio": caps_word_ratio,
    }


def extract_structural(texts):
    return pd.DataFrame([_structural_row(t) for t in texts])


if __name__ == "__main__":
    OUT_DIR = "Data/struct"
    os.makedirs(OUT_DIR, exist_ok=True)

    # ---- Load raw text ----
    train_raw = pd.read_csv('Data/dataset_main.csv', keep_default_na=False)
    temporal_raw = pd.read_csv('Data/test_temporal.csv', keep_default_na=False)
    obfuscated_raw = pd.read_csv('Data/test_obfuscated.csv', keep_default_na=False)
    notinject_raw = pd.read_csv('Data/test_notinject.csv', keep_default_na=False)

    # ---- Load existing engineered features ----
    train_feat = pd.read_csv('Data/features_main.csv', keep_default_na=False)
    temporal_feat = pd.read_csv('Data/features_test_temporal.csv', keep_default_na=False)
    obfuscated_feat = pd.read_csv('Data/features_test_obfuscated.csv', keep_default_na=False)
    notinject_feat = pd.read_csv('Data/features_test_notinject.csv', keep_default_na=False)

    assert len(train_raw) == len(train_feat)
    assert len(temporal_raw) == len(temporal_feat)
    assert len(obfuscated_raw) == len(obfuscated_feat)
    assert len(notinject_raw) == len(notinject_feat)

    # ---- Compute structural features ----
    train_struct = extract_structural(train_raw['text'].tolist())
    temporal_struct = extract_structural(temporal_raw['text'].tolist())
    obfuscated_struct = extract_structural(obfuscated_raw['text'].tolist())
    notinject_struct = extract_structural(notinject_raw['text'].tolist())

    # ---- Combine with existing engineered features ----
    def combine(feat_df, struct_df):
        return pd.concat([feat_df.reset_index(drop=True), struct_df.reset_index(drop=True)], axis=1)

    train_combined = combine(train_feat, train_struct)
    temporal_combined = combine(temporal_feat, temporal_struct)
    obfuscated_combined = combine(obfuscated_feat, obfuscated_struct)
    notinject_combined = combine(notinject_feat, notinject_struct)

    train_combined.to_csv(f'{OUT_DIR}/features_main_struct.csv', index=False)
    temporal_combined.to_csv(f'{OUT_DIR}/features_test_temporal_struct.csv', index=False)
    obfuscated_combined.to_csv(f'{OUT_DIR}/features_test_obfuscated_struct.csv', index=False)
    notinject_combined.to_csv(f'{OUT_DIR}/features_test_notinject_struct.csv', index=False)

    print(f"{OUT_DIR}/features_main_struct.csv: {train_combined.shape}")
    print(f"{OUT_DIR}/features_test_temporal_struct.csv: {temporal_combined.shape}")
    print(f"{OUT_DIR}/features_test_obfuscated_struct.csv: {obfuscated_combined.shape}")
    print(f"{OUT_DIR}/features_test_notinject_struct.csv: {notinject_combined.shape}")

features_main_struct.csv: (13489, 33)
features_test_temporal_struct.csv: (485, 33)
features_test_obfuscated_struct.csv: (1800, 35)
features_test_notinject_struct.csv: (339, 33)
